In [69]:
import pandas as pd
import time
import os
from deep_translator import GoogleTranslator

In [70]:
df = pd.read_csv("../data/processed/train_master.csv")

In [71]:
CHECKPOINT_PATH = "../data/processed/translation_checkpoint.csv"
OUTPUT_PATH     = '../data/processed/train_master_vi.csv'

DELAY_SECONDS   = 0.5
CHECKPOINT_EVERY = 500
MAX_CHARS       = 4900
print(f'Tổng số dòng cần dịch: {len(df):,}')
print(f'Phân phối nhãn:\n{df["label_unsafe"].value_counts()}')


Tổng số dòng cần dịch: 59,172
Phân phối nhãn:
label_unsafe
0    29586
1    29586
Name: count, dtype: int64


In [72]:
if os.path.exists(CHECKPOINT_PATH):
    df_done = pd.read_csv(CHECKPOINT_PATH)
    start_row = len(df_done)
    print(f'Phát hiện checkpoint! Tiếp tục từ dòng {start_row:,} / {len(df):,}')
else:
    df_done = pd.DataFrame(columns=['prompt', 'label_unsafe'])
    start_row = 0
    print(f'Bắt đầu dịch mới từ dòng 0...')

buffer = []
error_count = 0

for idx, row in df.iloc[start_row:].iterrows():
    original_text = str(row['prompt'])

    try:
        text_to_translate = original_text[:MAX_CHARS]
        vi_text = GoogleTranslator(source='auto', target='vi').translate(text_to_translate)
        if vi_text is None:
            vi_text = original_text

    except Exception as e:
        vi_text = original_text
        error_count += 1
        if error_count % 50 == 0:
            print(f'Gặp {error_count} lỗi tích lũy. Lỗi mới nhất: {e}')

    buffer.append({'prompt': vi_text, 'label_unsafe': row['label_unsafe']})

    current_total = start_row + len(buffer)

    if len(buffer) % CHECKPOINT_EVERY == 0:
        df_buffer = pd.DataFrame(buffer)
        df_done = pd.concat([df_done, df_buffer], ignore_index=True)
        df_done.to_csv(CHECKPOINT_PATH, index=False)
        buffer = []

        progress = current_total / len(df) * 100
        print(f'[{progress:.1f}%] Đã dịch {current_total:,} / {len(df):,} dòng | Lỗi: {error_count}')

    time.sleep(DELAY_SECONDS)

if buffer:
    df_buffer = pd.DataFrame(buffer)
    df_done = pd.concat([df_done, df_buffer], ignore_index=True)
    df_done.to_csv(CHECKPOINT_PATH, index=False)

print(f'\HOÀN THÀNH! Tổng lỗi gặp phải: {error_count}')
print(f'   Tổng dòng đã dịch: {len(df_done):,}')

Phát hiện checkpoint! Tiếp tục từ dòng 52,000 / 59,172
[88.7%] Đã dịch 52,500 / 59,172 dòng | Lỗi: 0
[88.7%] Đã dịch 52,500 / 59,172 dòng | Lỗi: 1
[88.7%] Đã dịch 52,500 / 59,172 dòng | Lỗi: 1
[88.7%] Đã dịch 52,500 / 59,172 dòng | Lỗi: 2
[88.7%] Đã dịch 52,500 / 59,172 dòng | Lỗi: 3
[88.7%] Đã dịch 52,500 / 59,172 dòng | Lỗi: 5
[88.7%] Đã dịch 52,500 / 59,172 dòng | Lỗi: 7
[88.7%] Đã dịch 52,500 / 59,172 dòng | Lỗi: 9
[88.7%] Đã dịch 52,500 / 59,172 dòng | Lỗi: 12
[88.7%] Đã dịch 52,500 / 59,172 dòng | Lỗi: 14
[88.7%] Đã dịch 52,500 / 59,172 dòng | Lỗi: 15
[88.7%] Đã dịch 52,500 / 59,172 dòng | Lỗi: 16
[88.7%] Đã dịch 52,500 / 59,172 dòng | Lỗi: 18
[88.7%] Đã dịch 52,500 / 59,172 dòng | Lỗi: 18
\HOÀN THÀNH! Tổng lỗi gặp phải: 18
   Tổng dòng đã dịch: 59,172


In [73]:
df_final = pd.read_csv(CHECKPOINT_PATH)

print(f'Tổng số dòng: {len(df_final):,}')
print(f'Phân phối nhãn sau dịch:\n{df_final["label_unsafe"].value_counts()}')
print(f'\nVí dụ 3 dòng đầu sau khi dịch:')
display(df_final.head(3))


df_final.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
print(f'\nĐã lưu file kết quả tại: {OUTPUT_PATH}')
print('   Hãy upload file này lên Google Drive!')

Tổng số dòng: 59,172
Phân phối nhãn sau dịch:
label_unsafe
0    29586
1    29586
Name: count, dtype: int64

Ví dụ 3 dòng đầu sau khi dịch:


,prompt,label_unsafe
0,Có thể áp dụng trình đổ bóng phân đoạn cho toà...,0
1,Tại sao các dịch vụ chuyển tiếp dành cho nạn n...,1
2,Dấu chân môi trường hàng năm của ChatGPT là gì...,0



Đã lưu file kết quả tại: ../data/processed/train_master_vi.csv
   Hãy upload file này lên Google Drive!
